In [ ]:

import sys
sys.path.append('../../utils')
import utility
import math
import itertools
import random
import numpy as np
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from openai import OpenAI
from openai.lib._parsing._completions import type_to_response_format_param
import json
import evaluate
import re
from z3 import *
from typing import Literal
# Use the MALLS code at https://github.com/gblackout/LogicLLaMA/blob/main/metrics.py
sys.path.append('../../utils/LogicLLaMA-main')
from fol_parser import msplit, parse_text_FOL_to_tree, VecRuleEvaluator

seeds_considered = [3, 12, 107, 85, 26]

In [ ]:
bleu = evaluate.load('bleu')
FOL_tokenizer = lambda x: msplit(x)[0]

def compute_FOL_bleu(pred_seq: str, true_seq: str):
        min_len = min(map(lambda x: len(FOL_tokenizer(x)), [pred_seq, true_seq]))
        res = bleu.compute(predictions=[pred_seq], references=[[true_seq]], 
                                tokenizer=FOL_tokenizer, max_order=min(4, min_len))
        return res['bleu']

def compute_LE(pred_text_FOL: str, true_text_FOL: str):
        true_root, pred_root = parse_text_FOL_to_tree(true_text_FOL), parse_text_FOL_to_tree(pred_text_FOL)
        
        # parsing true FOL should never fail
        assert true_root is not None, 'failed parsing true text FOL %s' % true_text_FOL
        
        # parsing pred FOL can fail if model produces invalid rule, in which case, LE score is 0
        if pred_root is None:
            return 0., None, None
        
        # if both parsed successfully, then compute LE score
        score, true_inputs, binded_pred_inputs = \
            VecRuleEvaluator.find_best_LE_score(
                true_root,
                pred_root,
                soft_binding=True,
                greedy_match=True,
                top_n=1000
            )
        return score, true_inputs, binded_pred_inputs

In [ ]:
# For each formula, tokenize it and transform it in the pre-order format like in MALLS.
def pre_order(prediction, ground_truth):
    '''Takes in input the string of the formula written in usual FOL and outputs it in the SMTLIB format'''
    try:   
        y = prediction
        x = ground_truth
        # - put in a nicer form to be parsed:
        # replace [ or ] with ( or )
        y = y.replace('[', '(')
        y = y.replace(']', ')')
        # - put a space after the quantifiers and remove the space between the quantifier and the varibale
        y = y.replace('∀ ', '∀')
        y = y.replace('∃ ', '∃')
        # - add a space after the quantified variable
        pattern = r"∀([a-zA-Z]+)\("
        replacement = r"∀\1 ("
        y = re.sub(pattern, replacement, y)

        pattern = r"∃([a-zA-Z]+)\("
        replacement = r"∃\1 ("
        y = re.sub(pattern, replacement, y)
        # remove final dots or comma for ending 
        if y[-1] == ',' or y[-1] == '.':
            y = y[:-1]
        
        # Order the constant, and predicate symbols from the longer to the shorter (in this way we don't have problem if we have two constants like 'son' and 'jamson')
        r, c1, v1 = utility.extract_rel_const_var(x)
        r2, c2, v2 = utility.extract_rel_const_var(y)
        c = c1.union(c2)
        v = v1.union(v2)
        r.update(r2)


        list_constants = sorted(list(c), key=len, reverse=True)

        unary_symbols = [rel for rel in r.keys() if r[rel] == 1]
        binary_symbols = [rel for rel in r.keys() if r[rel] == 2]
        ternary_symbols = [rel for rel in r.keys() if r[rel] == 3]
        
        
        list_replacements = sorted(list_constants + unary_symbols + binary_symbols + ternary_symbols, key=len, reverse=True)
    
        # for item in list_replacements:
        #     if item in list_constants:
        #         index = list_constants.index(item) + 1
        #         x = x.replace(item, 'A' + str(index))
        #         y = y.replace(item, 'A' + str(index))
        #     if item in unary_symbols:
        #         index = unary_symbols.index(item) + 1
        #         x = x.replace(item, 'U'+str(index))
        #         y = y.replace(item, 'U'+str(index))
        #     if item in binary_symbols:
        #         index = binary_symbols.index(item) + 1
        #         x = x.replace(item, 'B'+str(index)) 
        #         y = y.replace(item, 'B'+str(index))
        #     if item in ternary_symbols:
        #         index = ternary_symbols.index(item) + 1
        #         x = x.replace(item, 'T'+str(index))
        #         y = y.replace(item, 'T'+str(index))

        # Build a lookup dict of replacements
        replacements = {}
        for item in list_replacements:
            if item in list_constants:
                index = list_constants.index(item) + 1
                replacements[item] = 'A' + str(index)
            elif item in unary_symbols:
                index = unary_symbols.index(item) + 1
                replacements[item] = 'U' + str(index)
            elif item in binary_symbols:
                index = binary_symbols.index(item) + 1
                replacements[item] = 'B' + str(index)
            elif item in ternary_symbols:
                index = ternary_symbols.index(item) + 1
                replacements[item] = 'T' + str(index)

        # Compile one regex that matches any key
        pattern = re.compile('|'.join(map(re.escape, replacements.keys())))

        # Replace using the mapping
        def replace_all(match):
            return replacements[match.group(0)]

        x_new = pattern.sub(replace_all, x)
        y_new = pattern.sub(replace_all, y)
        

        gt = utility.parse_formula_SMTLIB_universal(x_new)
        pred = utility.parse_formula_SMTLIB_universal(y_new)
        return gt,pred
            
    except:
        print('Exception in parsing')
        return ground_truth, prediction

# FOLIO

In [ ]:
folio_dict = utility.make_jsonl_into_list_dictionaries('../../datasets/folio-train-clean.jsonl')
NL_sentences = {}
FOL_formulas = {}
story_id_already_seen = set()
for item in folio_dict:
    story_id = item['story_id']
    if story_id not in story_id_already_seen:
        story_id_already_seen.add(story_id)
        NL_sentences[story_id] = item['premises']
        FOL_formulas[story_id] = item['premises-FOL']
# remove the story 236 because logically problematic (PValue doesn't mantain the same arity in the FOL translations)
NL_sentences.pop(236)
FOL_formulas.pop(236)


        
# Delete the sentences in the stories that have the Xor symbol ⊕ (it is treated inconsistently in the dataset)
for i in range(2):
    for story_id in FOL_formulas.keys():
        for sentence in FOL_formulas[story_id]:
            if '⊕' in sentence:
                index = FOL_formulas[story_id].index(sentence)
                FOL_formulas[story_id].pop(index)
                NL_sentences[story_id].pop(index)


In [ ]:
# Translate in the new formalism (rename of predicate and constant symbols) the ground truths and the predictions
#model = 'gpt-4o-mini'
#model = 'o3-mini'
model = 'Qwen3-8B_think'
#model = 'Qwen3-30B-A3B_think'
clean_output = utility.load_pickle(f'../Logical Translation/Results/Clean Results/FOLIO_log_transl_{model}_seed.pkl')

predictions = {}
FOL_formulas_for_Z3_FOLIO = {}
predictions_for_Z3_FOLIO = {}
list_exception = {}
list_exception_use_equality = {}

for seed in seeds_considered:
    predictions[seed] = {}
    FOL_formulas_for_Z3_FOLIO[seed] = {}
    predictions_for_Z3_FOLIO[seed] = {}
    list_exception[seed] = []
    list_exception_use_equality[seed] = []


for seed in seeds_considered:
    for story_id in NL_sentences.keys():
        predictions[seed][story_id] = ['' for i in NL_sentences[story_id]]
        FOL_formulas_for_Z3_FOLIO[seed][story_id] = ['' for i in NL_sentences[story_id]]
        predictions_for_Z3_FOLIO[seed][story_id] = ['' for i in NL_sentences[story_id]]
    for line in clean_output[seed]:
        story_id, number_instance, answer = int(line['story_id']), int(line['number_instance']), line['answer']
        predictions[seed][story_id][number_instance] = answer


    for story_id in FOL_formulas.keys():
        for i, formula in enumerate(FOL_formulas[story_id]):
            x = pre_order(formula, predictions[seed][story_id][i])
            FOL_formulas_for_Z3_FOLIO[seed][story_id][i] = x[0]
            predictions_for_Z3_FOLIO[seed][story_id][i] = x[1]
            

    print('Seed:', seed)
    print('list_exception_without_eq = ', [i for i in list_exception[seed] if i not in list_exception_use_equality[seed]])
    print('length list_exception_without_eq = ', len([i for i in list_exception[seed] if i not in list_exception_use_equality[seed]]))
    print('instances in which equality is used', list_exception_use_equality[seed])
    print('length list_exception with equality = ', len(list_exception_use_equality[seed]))
    print('_'*20)

In [ ]:

correct_translations = {}
list_correct_positions_FOLIO = {}
list_incorrect_positions_FOLIO = {}
list_done = {}
score_BLEU_FOLIO = {}
score_LE_FOLIO = {}
avg_LE, avg_BLEU = {}, {}

for seed in seeds_considered:
    correct_translations[seed] = {}
    list_correct_positions_FOLIO[seed] = {}
    list_incorrect_positions_FOLIO[seed] = {}
    list_done[seed] = {}
    score_LE_FOLIO[seed] = {}
    score_BLEU_FOLIO[seed] = {}

In [ ]:
# Remove the item that are in list_ep

for seed in seeds_considered:
    for story_id in NL_sentences.keys():
        correct_translations[seed][story_id] = ['' for i in NL_sentences[story_id]]
        score_BLEU_FOLIO[seed][story_id] = ['' for i in NL_sentences[story_id]]
        score_LE_FOLIO[seed][story_id] = ['' for i in NL_sentences[story_id]]
        list_correct_positions_FOLIO[seed][story_id] = []
        list_incorrect_positions_FOLIO[seed][story_id] = []
        list_done[seed][story_id] = []
        for i, sent in enumerate(NL_sentences[story_id]):
            key = f'{story_id}_{i}'
            if key in list_exception[seed]:
                if key in list_exception_use_equality[seed]:
                    correct_translations[seed][story_id][i] = 'equality_used'
                else:
                    correct_translations[seed][story_id][i] = 'parsing_error'
            else:
                pred = predictions_for_Z3_FOLIO[seed][story_id][i]
                correct_translations[seed][story_id][i] = utility.check_equivalence_with_parsed_formulas(pred, FOL_formulas_for_Z3_FOLIO[seed][story_id][i])
            
                if correct_translations[seed][story_id][i] == True:
                    list_correct_positions_FOLIO[seed][story_id].append(i)
                elif correct_translations[seed][story_id][i] == False:
                    list_incorrect_positions_FOLIO[seed][story_id].append(i)
                
            list_done[seed][story_id].append(i)

 


    # Now let's compute the LE/BLEU score when the translations are correct or wrong
    bleu_if_wrong = 0
    bleu_if_correct = 0
    le_if_wrong = 0
    le_if_correct = 0
    count_wrong = 0
    count_correct = 0
    exceptions = 0
            

    for story_id in FOL_formulas.keys():
        for i in list_incorrect_positions_FOLIO[seed][story_id]:
            pred = str(predictions_for_Z3_FOLIO[seed][story_id][i])
            ref = str(FOL_formulas_for_Z3_FOLIO[seed][story_id][i])
            exceptions_wrong = 0
            try:
                count_wrong += 1
                score_BLEU_FOLIO[seed][story_id][i] = compute_FOL_bleu(pred, ref)
                score_LE_FOLIO[seed][story_id][i] = compute_LE(predictions[seed][story_id][i], FOL_formulas[story_id][i])[0]
                bleu_if_wrong += score_BLEU_FOLIO[seed][story_id][i]
                le_if_wrong += score_LE_FOLIO[seed][story_id][i]
            except:
                exceptions_wrong += 1
                

        for i in list_correct_positions_FOLIO[seed][story_id]:
            pred = str(predictions_for_Z3_FOLIO[seed][story_id][i])
            ref = str(FOL_formulas_for_Z3_FOLIO[seed][story_id][i])
            exceptions_correct = 0
            try:
                count_correct += 1
                score_BLEU_FOLIO[seed][story_id][i] = compute_FOL_bleu(pred, ref)
                x = compute_LE(predictions[seed][story_id][i], FOL_formulas[story_id][i])
                score_LE_FOLIO[seed][story_id][i] = x[0]
                bleu_if_correct += score_BLEU_FOLIO[seed][story_id][i]
                le_if_correct += score_LE_FOLIO[seed][story_id][i]
            except:
                exceptions_correct += 1
    
    # Compute the avg and stdv of BLEU and LE score
    avg = 0
    counter = 0
    for story_id in score_BLEU_FOLIO[seed].keys():
        for val in score_BLEU_FOLIO[seed][story_id]:
            if isinstance(val, float):
                avg += val
            else:
                print(f'Error in {story_id}:', val)
            counter += 1
    avg_BLEU[seed] = avg/counter

    avg = 0
    counter = 0
    for story_id in score_LE_FOLIO[seed].keys():
        for val in score_LE_FOLIO[seed][story_id]:
            if isinstance(val, float):
                avg += val
            else:
                print(f'Error in {story_id}:', val)
            counter += 1
    avg_LE[seed] = avg/counter

    print('Model: ', model)
    print('Seed: ', seed)
    print('wrong', count_wrong)
    print('correct', count_correct)
    print('exceptions', exceptions)
    print('LE score:', avg_LE[seed])
    print('BLEU score:', avg_BLEU[seed])
    utility.save_pickle(avg_LE[seed], f'LE_{seed}_30.pkl')
    utility.save_pickle(avg_BLEU[seed], f'BLEU_{seed}_30.pkl')
    # print('LE score when correct (this should be 1): ', le_if_correct/count_correct)
    # print('LE score when incorrect: ', le_if_wrong/count_wrong)
    # print('BLEU score when correct: ', bleu_if_correct/count_correct)
    # print('BLEU score when incorrect: ', bleu_if_wrong/count_wrong)

print('____')
avgle = sum(avg_LE.values())/len(seeds_considered)
print('Avg_LE: ', avgle)
error_LE = [(i - avgle)**2 for i in avg_LE.values()]
print('sdv_LE', math.sqrt(sum(error_LE)/len(seeds_considered)))

avgbleu = sum(avg_BLEU.values())/len(seeds_considered)
print('Avg_BLEU: ', avgbleu)
error_BLEU = [(i - avgle)**2 for i in avg_BLEU.values()]
print('sdv_BLEU', math.sqrt(sum(error_BLEU)/len(seeds_considered)))

In [ ]:
avg_BLEU = {}
avg_BLEU[3], avg_BLEU[12] = utility.load_pickle(f'BLEU_3.pkl'), utility.load_pickle(f'BLEU_12.pkl')
print(avg_BLEU)

avg_LE = {}
avg_LE[3], avg_LE[12]= utility.load_pickle(f'LE_3.pkl'), utility.load_pickle(f'LE_12.pkl'),
print(avg_LE)


In [ ]:
from tqdm import tqdm

# Remove the item that are in list_ep

correct_translations = {}
list_correct_positions_FOLIO = {}
list_incorrect_positions_FOLIO = {}
list_done = {}
score_BLEU_FOLIO = {}
score_LE_FOLIO = {}

for seed in seeds_considered:
    correct_translations[seed] = {}
    list_correct_positions_FOLIO[seed] = {}
    list_incorrect_positions_FOLIO[seed] = {}
    list_done[seed] = {}
    score_LE_FOLIO[seed] = {}
    score_BLEU_FOLIO[seed] = {}

seed_map_all_binaries = {}
seed_map_all_bleus = {}
seed_map_all_les = {}


for seed in tqdm(seeds_considered):
    all_binaries = []
    all_bleus = []
    all_les = []
    for story_id in NL_sentences.keys():
        correct_translations[seed][story_id] = ['' for i in NL_sentences[story_id]]
        score_BLEU_FOLIO[seed][story_id] = ['' for i in NL_sentences[story_id]]
        score_LE_FOLIO[seed][story_id] = ['' for i in NL_sentences[story_id]]
        list_correct_positions_FOLIO[seed][story_id] = []
        list_incorrect_positions_FOLIO[seed][story_id] = []
        list_done[seed][story_id] = []
        for i, sent in enumerate(NL_sentences[story_id]):
            key = f'{story_id}_{i}'
            if key in list_exception[seed]:
                if key in list_exception_use_equality[seed]:
                    correct_translations[seed][story_id][i] = 'equality_used'
                else:
                    correct_translations[seed][story_id][i] = 'parsing_error'
            else:
                pred = predictions_for_Z3_FOLIO[seed][story_id][i]
                correct_translations[seed][story_id][i] = utility.check_equivalence_with_parsed_formulas(pred, FOL_formulas_for_Z3_FOLIO[seed][story_id][i])
            
                if correct_translations[seed][story_id][i] == True:
                    list_correct_positions_FOLIO[seed][story_id].append(i)
                elif correct_translations[seed][story_id][i] == False:
                    list_incorrect_positions_FOLIO[seed][story_id].append(i)


                try:
                    predtry= str(predictions_for_Z3_FOLIO[seed][story_id][i])
                    reftry = str(FOL_formulas_for_Z3_FOLIO[seed][story_id][i])
                    bleutry = compute_FOL_bleu(predtry, reftry)
                    letry = compute_LE(predictions[seed][story_id][i], FOL_formulas[story_id][i])[0]
                except Exception as e:
                    # Optional: log the error if you want to know what failed
                    print(f"Error computing BLEU/LE for {seed}-{story_id}-{i}: {e}")
                    continue  # skip this iteration safely
                else:
                    all_binaries.append(correct_translations[seed][story_id][i])
                    all_bleus.append(bleutry)
                    all_les.append(letry) 
                
            list_done[seed][story_id].append(i)

        seed_map_all_binaries[seed] = all_binaries
        seed_map_all_bleus[seed] = all_bleus
        seed_map_all_les[seed] = all_les



In [ ]:
from scipy.stats import pointbiserialr
bleu_corrs = []
le_corrs = []
for s in seed_map_all_binaries.keys():
    print("seed", s)
    bleu_corrs.append(pointbiserialr(seed_map_all_binaries[s], seed_map_all_bleus[s]))
    le_corrs.append(pointbiserialr(seed_map_all_binaries[s], seed_map_all_les[s]))
    print("bleu", bleu_corrs[-1])
    print("le", le_corrs[-1])

print("")

print("bleu stats", np.min([x[0] for x in bleu_corrs]), np.max([x[0] for x in bleu_corrs]), np.mean([x[0] for x in bleu_corrs]), np.std([x[0] for x in bleu_corrs]))
print("le stats", np.min([x[0] for x in le_corrs]), np.max([x[0] for x in le_corrs]), np.mean([x[0] for x in le_corrs]), np.std([x[0] for x in le_corrs]))



# 4o
# bleu stats 0.8012781673453031 0.8150509861008834 0.8079706153546317 0.0057545347423636276
# le stats 0.6098470152864677 0.6264685822382711 0.6164561754000191 0.005891487735234573

# o3
# bleu stats 0.7922337261559789 0.8009707766467916 0.7957008330713128 0.0034660444438694075
# le stats 0.6200551745772142 0.6542587577368026 0.64018613727714 0.011139767071421756

## Stanford

In [ ]:
FOL_sentences = [x for x in utility.make_txt_into_list('../../datasets/FOLsentence.txt')]
number_instances = len(FOL_sentences)
ground_truths = utility.make_txt_into_list('../../datasets/ground-truth.txt')

number_instances = len(FOL_sentences)

In [ ]:
# model = 'gpt-4o-mini'
# clean_output = utility.load_pickle(f'../Logical Translation/Results/Clean Results/Stanford_log_transl_{model}_seed.pkl')
# print(clean_output[26][0])

# for seed in seeds_considered:
#     for i, line in enumerate(clean_output[seed]):
#         line['number_instance'] = i

# utility.save_pickle(clean_output, f'../Logical Translation/Results/Clean Results/Stanford_log_transl_{model}_seed.pkl')

In [ ]:
# Translate in the new formalism (rename of predicate and constant symbols) the ground truths and the predictions
#model = 'gpt-4o-mini'
#model = 'o3-mini'
#model = 'Qwen3-8B_think'
model = 'Qwen3-30B-A3B_think'
clean_output = utility.load_pickle(f'../Logical Translation/Results/Clean Results/Stanford_log_transl_{model}_seed.pkl')

predictions = {}
FOL_formulas_for_Z3 = {}
predictions_for_Z3 = {}
list_exception = {}
list_exception_use_equality = {}

for seed in seeds_considered:
    predictions[seed] = []
    FOL_formulas_for_Z3[seed] = []
    predictions_for_Z3[seed] = []
    list_exception[seed] = []
    list_exception_use_equality[seed] = []


for seed in seeds_considered:
    predictions[seed] = ['' for i in FOL_sentences]
    FOL_formulas_for_Z3[seed] = ['' for i in FOL_sentences]
    predictions_for_Z3[seed] = ['' for i in FOL_sentences]
    for line in clean_output[seed]:
        number_instance, answer =  int(line['number_instance']), line['answer']
        predictions[seed][number_instance] = answer



    for i, formula in enumerate(FOL_sentences):
        x = pre_order(formula, predictions[seed][i])
        FOL_formulas_for_Z3[seed][i] = x[0]
        predictions_for_Z3[seed][i] = x[1]
            

    print('Seed:', seed)
    print('list_exception_without_eq = ', [i for i in list_exception[seed] if i not in list_exception_use_equality[seed]])
    print('length list_exception_without_eq = ', len([i for i in list_exception[seed] if i not in list_exception_use_equality[seed]]))
    print('instances in which equality is used', list_exception_use_equality[seed])
    print('length list_exception with equality = ', len(list_exception_use_equality[seed]))
    print('_'*20)

In [ ]:
# Remove the item that are in list_ep


correct_translations = {}
list_correct_positions = {}
list_incorrect_positions = {}
list_done = {}
score_BLEU_Stanford = {}
score_LE_Stanford = {}
avg_BLEU, avg_LE = {},{}

for seed in seeds_considered:
    correct_translations[seed] = []
    list_correct_positions[seed] = []
    list_incorrect_positions[seed] = []
    list_done[seed] = []
    score_LE_Stanford[seed] = []
    score_BLEU_Stanford[seed] = []

for seed in seeds_considered:

    correct_translations[seed] = ['' for i in FOL_sentences]
    score_BLEU_Stanford[seed] = ['' for i in FOL_sentences]
    score_LE_Stanford[seed] = ['' for i in FOL_sentences]
    list_correct_positions[seed] = []
    list_incorrect_positions[seed] = []
    list_done[seed] = []
    for i, sent in enumerate(FOL_sentences):
        key = f'{i}'
        if key in list_exception[seed]:
            if key in list_exception_use_equality[seed]:
                correct_translations[seed][i] = 'equality_used'
            else:
                correct_translations[seed][i] = 'parsing_error'
        else:
            pred = predictions_for_Z3[seed][i]
            correct_translations[seed][i] = utility.check_equivalence_with_parsed_formulas(pred, FOL_formulas_for_Z3[seed][i])
        
            if correct_translations[seed][i] == True:
                list_correct_positions[seed].append(i)
            elif correct_translations[seed][i] == False:
                list_incorrect_positions[seed].append(i)
            
        list_done[seed].append(i)


    # Now let's compute the LE/BLEU score when the translations are correct or wrong
    bleu_if_wrong = 0
    bleu_if_correct = 0
    le_if_wrong = 0
    le_if_correct = 0
    count_wrong = 0
    count_correct = 0


    
    for i in list_incorrect_positions[seed]:
        pred = str(predictions_for_Z3[seed][i])
        ref = str(FOL_formulas_for_Z3[seed][i])
        exceptions_wrong = 0
        try:
            count_wrong += 1
            score_BLEU_Stanford[seed][i] = compute_FOL_bleu(pred, ref)
            score_LE_Stanford[seed][i] = compute_LE(predictions[seed][i], FOL_sentences[i])[0]
            bleu_if_wrong += score_BLEU_Stanford[seed][i]
            le_if_wrong += score_LE_Stanford[seed][i]
        except:
            exceptions_wrong += 1
            

    for i in list_correct_positions[seed]:
        pred = str(predictions_for_Z3[seed][i])
        ref = str(FOL_formulas_for_Z3[seed][i])
        exceptions_correct = 0
        try:
            count_correct += 1
            score_BLEU_Stanford[seed][i] = compute_FOL_bleu(pred, ref)
            x = compute_LE(predictions[seed][i], FOL_sentences[i])
            score_LE_Stanford[seed][i] = x[0]
            bleu_if_correct += score_BLEU_Stanford[seed][i]
            le_if_correct += score_LE_Stanford[seed][i]
        except:
            exceptions_correct += 1
            


    # Compute the avg and stdv of BLEU and LE score
    avg = 0
    counter = 0
    for val in score_BLEU_Stanford[seed]:
        if isinstance(val, float):
            avg += val
        else:
            print(f'Error in:', val)
        counter += 1
    avg_BLEU[seed] = avg/counter

    avg = 0
    counter = 0
    for val in score_LE_Stanford[seed]:
        if isinstance(val, float):
            avg += val
        else:
            print(f'Error:', val)
        counter += 1
    avg_LE[seed] = avg/counter

    print('Model: ', model)
    print('Seed: ', seed)
    print('wrong', count_wrong)
    print('correct', count_correct)
    print('LE score:', avg_LE[seed])
    print('BLEU score:', avg_BLEU[seed] )
    # print('LE score when correct (this should be 1): ', le_if_correct/count_correct)
    # print('LE score when incorrect: ', le_if_wrong/count_wrong)
    # print('BLEU score when correct: ', bleu_if_correct/count_correct)
    # print('BLEU score when incorrect: ', bleu_if_wrong/count_wrong)

print('____')
avgle = sum(avg_LE.values())/len(seeds_considered)
print('Avg_LE: ', avgle)
error_LE = [(i - avgle)**2 for i in avg_LE.values()]
print('sdv_LE', math.sqrt(sum(error_LE)/len(seeds_considered)))

avgbleu = sum(avg_BLEU.values())/len(seeds_considered)
print('Avg_BLEU: ', avgbleu)
error_BLEU = [(i - avgle)**2 for i in avg_BLEU.values()]
print('sdv_BLEU', math.sqrt(sum(error_BLEU)/len(seeds_considered)))

In [ ]:
from tqdm import tqdm


correct_translations = {}
list_correct_positions = {}
list_incorrect_positions = {}
list_done = {}
score_BLEU_Stanford = {}
score_LE_Stanford = {}

seed_map_all_binaries = {}
seed_map_all_bleus = {}
seed_map_all_les = {}


for seed in seeds_considered:
    correct_translations[seed] = []
    list_correct_positions[seed] = []
    list_incorrect_positions[seed] = []
    list_done[seed] = []
    score_LE_Stanford[seed] = []
    score_BLEU_Stanford[seed] = []

for seed in tqdm(seeds_considered):

    all_binaries = []
    all_bleus = []
    all_les = []

    correct_translations[seed] = ['' for i in FOL_sentences]
    score_BLEU_Stanford[seed] = ['' for i in FOL_sentences]
    score_LE_Stanford[seed] = ['' for i in FOL_sentences]
    list_correct_positions[seed] = []
    list_incorrect_positions[seed] = []
    list_done[seed] = []
    for i, sent in enumerate(FOL_sentences):

        pred = predictions_for_Z3[seed][i]
        correct_translations[seed][i] = utility.check_equivalence_with_parsed_formulas(pred, FOL_formulas_for_Z3[seed][i])
    
        if correct_translations[seed][i] == True:
            list_correct_positions[seed].append(i)
        elif correct_translations[seed][i] == False:
            list_incorrect_positions[seed].append(i)

        try:
            predtry = str(predictions_for_Z3[seed][i])
            reftry = str(FOL_formulas_for_Z3[seed][i])
            bleutry = compute_FOL_bleu(predtry, reftry)
            le = compute_LE(predictions[seed][i], FOL_sentences[i])[0]
        except Exception as e:
            # Optional: log the error if you want to know what failed
            print(f"Error computing BLEU/LE for {seed}-{i}: {e}")
            continue  # skip this iteration safely
        else:
            all_binaries.append(correct_translations[seed][i])
            all_bleus.append(bleutry)
            all_les.append(le) 
            
        list_done[seed].append(i)

    seed_map_all_binaries[seed] = all_binaries
    seed_map_all_bleus[seed] = all_bleus
    seed_map_all_les[seed] = all_les

In [ ]:
from scipy.stats import pointbiserialr
bleu_corrs = []
le_corrs = []
for s in seed_map_all_binaries.keys():
    print("seed", s)
    bleu_corrs.append(pointbiserialr(seed_map_all_binaries[s], seed_map_all_bleus[s]))
    le_corrs.append(pointbiserialr(seed_map_all_binaries[s], seed_map_all_les[s]))
    print("bleu", bleu_corrs[-1])
    print("le", le_corrs[-1])

print("")

print("bleu stats", np.min([x[0] for x in bleu_corrs]), np.max([x[0] for x in bleu_corrs]), np.mean([x[0] for x in bleu_corrs]), np.std([x[0] for x in bleu_corrs]))
print("le stats", np.min([x[0] for x in le_corrs]), np.max([x[0] for x in le_corrs]), np.mean([x[0] for x in le_corrs]), np.std([x[0] for x in le_corrs]))



# 4o
# bleu stats 0.4914506250132909 0.7052590444682738 0.5648231090060227 0.07653386591462315
# le stats 0.4015260361864315 0.5658584706351796 0.4946288785532741 0.053963045286943845

# o3
# bleu stats 0.4465245750660146 0.5537674699821777 0.4913998346323324 0.0392963057642622
# le stats 0.4661878893515524 0.692144373568042 0.5828959604667228 0.0823958849787202

In [ ]:
# Compute AUC for Stanford
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
import matplotlib.pyplot as plt

# data
y_true = {}
le_score = {}
bleu_score = {}
weighted_score = {}

for seed in seeds_considered:
    y_true[seed] = []
    le_score[seed] = []
    bleu_score[seed] = []
    weighted_score[seed] = []

for seed in seeds_considered:
    for i in range(number_instances):
        if score_LE_Stanford[seed][i] != '':
            if i in list_correct_positions[seed]:
                y_true[seed].append(1)
                le_score[seed].append(score_LE_Stanford[seed][i])
                bleu_score[seed].append(score_BLEU_Stanford[seed][i])
                weighted_score[seed].append(0.7*score_LE_Stanford[seed][i] + 0.3*score_BLEU_Stanford[seed][i])
            elif i in list_incorrect_positions[seed]:
                y_true[seed].append(0)
                le_score[seed].append(score_LE_Stanford[seed][i])
                bleu_score[seed].append(score_BLEU_Stanford[seed][i])
                weighted_score[seed].append(0.7*score_LE_Stanford[seed][i] + 0.3*score_BLEU_Stanford[seed][i])
    
    print('Stanford 4o-mini')

    print('Seed', seed)
    # 1️⃣ Compute AUC
    auc_le = roc_auc_score(y_true[seed], le_score[seed])
    print(f"AUC_LE = {auc_le:.3f}")

    auc_bleu = roc_auc_score(y_true[seed], bleu_score[seed])
    print(f"AUC_BLEU = {auc_bleu:.3f}")

    auc_weighted = roc_auc_score(y_true[seed], weighted_score[seed])
    print(f"AUC_weighted = {auc_weighted:.3f}")


    # # Precision-Recall AUC
    # pr_auc_le = average_precision_score(y_true[seed], le_score[seed])
    # pr_auc_bleu = average_precision_score(y_true[seed], bleu_score[seed]) 
    # print(f"PR AUC LE = {pr_auc_le:.3f}")
    # print(f"PR AUC BLEU = {pr_auc_bleu:.3f}")


In [ ]:
# Compute AUC for FOLIO

from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
import matplotlib.pyplot as plt

# data
y_true = {}
le_score = {}
bleu_score = {}
weighted_score = {}

for seed in seeds_considered:
    y_true[seed] = []
    le_score[seed] = []
    bleu_score[seed] = []
    weighted_score[seed] = []



for seed in seeds_considered:
    for story_id in FOL_formulas.keys():
        for i in range(len(FOL_formulas[story_id])):
            if score_LE_FOLIO[seed][story_id][i] != '':
                if i in list_correct_positions_FOLIO[seed][story_id]:
                    y_true[seed].append(1)
                    le_score[seed].append(score_LE_FOLIO[seed][story_id][i])
                    bleu_score[seed].append(score_BLEU_FOLIO[seed][story_id][i])
                    weighted_score[seed].append(0.7*score_LE_FOLIO[seed][story_id][i] + 0.3*score_BLEU_FOLIO[seed][story_id][i])
                elif i in list_incorrect_positions_FOLIO[seed][story_id]:
                    y_true[seed].append(0)
                    le_score[seed].append(score_LE_FOLIO[seed][story_id][i])
                    bleu_score[seed].append(score_BLEU_FOLIO[seed][story_id][i])
                    weighted_score[seed].append(0.7*score_LE_FOLIO[seed][story_id][i] + 0.3*score_BLEU_FOLIO[seed][story_id][i])
                
    
    print('FOLIO, o3-mini')
    print('Seed', seed)
    # 1️⃣ Compute AUC
    auc_le = roc_auc_score(y_true[seed], le_score[seed])
    print(f"AUC_LE = {auc_le:.3f}")

    auc_bleu = roc_auc_score(y_true[seed], bleu_score[seed])
    print(f"AUC_BLEU = {auc_bleu:.3f}")

    auc_weighted = roc_auc_score(y_true[seed], weighted_score[seed])
    print(f"AUC weighted = {auc_weighted:.3f}")

    # 2️⃣ Compute ROC curve points
    #fpr, tpr, thresholds = roc_curve(y_true[seed], le_score[seed])


    
    
